# The Cold Start Problem in Recommender Systems

Companion notebook for the [Cold Start wiki page](https://ml-viz-ruby.vercel.app/wiki/cold-start-problem).

We implement content-based embedding projection for new items, popularity-based fallback, and the UCB1 exploration bonus for new items.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(8)

## 1 — Content-based projection for new items

In [ ]:
# Simulated embedding space (d=16) and content feature space (d_c=32)
d, d_c, n_items = 16, 32, 200
item_emb = rng.normal(size=(n_items, d))  # interaction-based embeddings (warm items)
item_emb /= np.linalg.norm(item_emb, axis=1, keepdims=True)
content_feats = rng.normal(size=(n_items, d_c))  # content features for all items

# Learn projection: W maps content features -> embedding space
# Using pseudo-inverse (in production: trained jointly with the rec model)
W_proj = np.linalg.lstsq(content_feats, item_emb, rcond=None)[0]  # (d_c, d)

def cold_start_embed(content):
    emb = content @ W_proj
    return emb / np.linalg.norm(emb)

# New item: content features only, no interaction history
new_item_content = rng.normal(size=d_c)
new_item_emb = cold_start_embed(new_item_content)

# Find similar warm items
sims = item_emb @ new_item_emb
top5 = np.argsort(-sims)[:5]
print("New item's most similar existing items:", top5.tolist())
print("Similarity scores:", sims[top5].round(4))

## 2 — UCB1 exploration bonus for new items

In [ ]:
class ItemUCB:
    def __init__(self, n_items):
        self.n = np.zeros(n_items)           # times shown
        self.mu = np.zeros(n_items)          # estimated CTR
        self.t = 0                           # total rounds

    def score(self, item_id, relevance_score, c=1.0):
        """UCB score = relevance + exploration bonus."""
        ucb_bonus = c * np.sqrt(np.log(self.t + 1) / (self.n[item_id] + 1))
        return relevance_score + ucb_bonus

    def update(self, item_id, clicked):
        self.n[item_id] += 1
        self.mu[item_id] = (self.mu[item_id] * (self.n[item_id]-1) + clicked) / self.n[item_id]
        self.t += 1

n_catalog = 100
ucb = ItemUCB(n_catalog)
relevance = rng.uniform(0.1, 0.9, n_catalog)   # model's base relevance scores

# Simulate 500 recommendation rounds
for t in range(500):
    # Score all items
    scores = np.array([ucb.score(i, relevance[i]) for i in range(n_catalog)])
    chosen = np.argmax(scores)
    # Simulate click (true CTR = relevance)
    clicked = float(rng.random() < relevance[chosen])
    ucb.update(chosen, clicked)

print(f"Most shown items: {np.argsort(-ucb.n)[:5].tolist()}")
print(f"Items never shown: {(ucb.n == 0).sum()}")
print(f"Estimated CTR for top item: {ucb.mu[np.argmax(ucb.mu)]:.3f}")
print(f"True CTR for top item:      {relevance[np.argmax(ucb.mu)]:.3f}")

## ✏️ Your turn

In [ ]:
def onboarding_item_selection(item_emb, k=10):
    """
    Select k items for an onboarding preference survey that maximally
    cover the embedding space diversity.
    
    Strategy: greedy farthest-point sampling — iteratively pick the item
    that is farthest from all already-selected items (maximizing min distance).
    
    item_emb: (N, d) normalized item embeddings
    Returns: list of k item indices
    """
    # TODO(you): start with a random item, then greedily add the item
    # most distant from all already-selected items
    return ...

selected = onboarding_item_selection(item_emb, k=10)
print(f"Selected {len(selected)} items for onboarding: {selected}")
# Verify diversity: mean pairwise distance should be high
pairs = [(i,j) for i in selected for j in selected if i<j]
mean_dist = np.mean([1 - item_emb[i] @ item_emb[j] for i,j in pairs])
print(f"Mean pairwise distance: {mean_dist:.4f} (random baseline ≈ {1 - item_emb[:10] @ item_emb[:10].T * 0 + 0.5:.2f})")

<details><summary>Solution</summary>

```python
def onboarding_item_selection(item_emb, k=10):
    N = len(item_emb)
    selected = [rng.integers(N).item()]
    for _ in range(k - 1):
        # Distance from each candidate to the nearest selected item
        min_dists = np.array([
            1 - max(item_emb[i] @ item_emb[s] for s in selected)
            for i in range(N)
        ])
        min_dists[selected] = -1   # don't re-select
        selected.append(int(np.argmax(min_dists)))
    return selected
```
</details>